# Module 27 — Exercise 1: SQLite Fundamentals, Parameterized Queries, and Transactions

In this exercise, you will master low-level database operations with Python's standard `sqlite3` driver: schema creation, parameterized SQL statements to eliminate SQL injection, transactions, and row factory mapping.

| Detail | Value |
|---|---|
| **Time** | 35 minutes |
| **Prerequisites** | Module 27 README, Module 19 |



## 1. Parameterized Queries vs String Formatting

**Rule:** Never format variables directly into SQL queries with f-strings or `%s`. Always use parameter placeholders (`?` in sqlite3).


In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, username TEXT, balance REAL)")

# Safe parameterized insertion:
conn.execute("INSERT INTO users (username, balance) VALUES (?, ?)", ("alice", 150.0))
conn.commit()

row = conn.execute("SELECT username, balance FROM users WHERE username = ?", ("alice",)).fetchone()
print("Retrieved user:", row)



## 2. Transactions and Atomic Commit / Rollback

The context manager `with conn:` handles transaction boundaries automatically: committing if the block succeeds and rolling back if an unhandled exception occurs.


In [ ]:
# Atomic Transfer Demo
try:
    with conn:
        conn.execute("UPDATE users SET balance = balance - 50 WHERE username = ?", ("alice",))
        # Deliberate error before crediting recipient
        raise ValueError("Simulated network crash during transfer!")
        conn.execute("UPDATE users SET balance = balance + 50 WHERE username = ?", ("bob",))
except ValueError:
    print("Transfer aborted and rolled back cleanly!")

# Alice's balance remains unchanged at 150.0
row = conn.execute("SELECT balance FROM users WHERE username = ?", ("alice",)).fetchone()
print("Alice balance after rollback:", row[0])



# Your turn


### Task 1: Create a Product Catalog Table and Insert Function

Implement `init_db(conn)` to create a `products` table with columns: `id` (INTEGER PRIMARY KEY AUTOINCREMENT), `name` (TEXT NOT NULL UNIQUE), `price` (REAL NOT NULL), and `stock` (INTEGER NOT NULL DEFAULT 0).
Implement `add_product(conn, name, price, stock)` using parameterized SQL.


In [ ]:
# ANSWER 1
def init_db(conn: sqlite3.Connection) -> None:
    with conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS products (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL UNIQUE,
                price REAL NOT NULL,
                stock INTEGER NOT NULL DEFAULT 0
            )
        """)

def add_product(conn: sqlite3.Connection, name: str, price: float, stock: int = 0) -> int:
    with conn:
        cur = conn.execute(
            "INSERT INTO products (name, price, stock) VALUES (?, ?, ?)",
            (name, price, stock)
        )
        return cur.lastrowid



### Task 2: Atomic Inventory Deduction

Implement `deduct_inventory(conn, product_id, quantity)` which checks if `stock >= quantity`. If sufficient stock exists, deduct `quantity` and return `True`. If insufficient, make no changes and return `False`. Ensure the entire operation is executed atomically within a transaction.


In [ ]:
# ANSWER 2
def deduct_inventory(conn: sqlite3.Connection, product_id: int, quantity: int) -> bool:
    with conn:
        row = conn.execute("SELECT stock FROM products WHERE id = ?", (product_id,)).fetchone()
        if not row or row[0] < quantity:
            return False
        conn.execute("UPDATE products SET stock = stock - ? WHERE id = ?", (quantity, product_id))
        return True



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

test_conn = sqlite3.connect(":memory:")
init_db(test_conn)
pid = add_product(test_conn, "Widget Pro", 29.99, 10)

success_deduct = deduct_inventory(test_conn, pid, 4)
stock_after = test_conn.execute("SELECT stock FROM products WHERE id = ?", (pid,)).fetchone()[0]

fail_deduct = deduct_inventory(test_conn, pid, 10)  # only 6 left
stock_final = test_conn.execute("SELECT stock FROM products WHERE id = ?", (pid,)).fetchone()[0]

results = [
    check(pid == 1, "Task 1: Product inserted with ID 1"),
    check(success_deduct is True and stock_after == 6, "Task 2: Deducted 4 units, 6 remaining"),
    check(fail_deduct is False and stock_final == 6, "Task 2: Overdraft rejected and stock unchanged"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

